# Portfolio Project #5
## Logistics & Fleet Managment

In [4]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

In [5]:
# --- 1. GENERATE DRIVERS (100 records) ---
# Schema: driver_id (Primary Key), first_name, last_name, license_class, hire_date, status (Active/Inactive)
print('Generating Drivers...')
drivers_data = []
license_classes = ['Class A', 'Class B', 'Class C']
statuses = ['Active', 'Active', 'Active', 'Inactive'] # Weight towards Active

for driver_id in range(1, 101):
    drivers_data.append({
        'driver_id': driver_id,
        'first_name': fake.first_name(),
        'last_name': fake.last_name(),
        'license_class': random.choice(license_classes),
        'hire_date': fake.date_between(start_date='-5y', end_date='today'),
        'status': random.choice(statuses)
    })
df_drivers = pd.DataFrame(drivers_data)

Generating Drivers...


In [6]:
# --- 2. GENERATE VEHICLES (50 records) ---
# Schema: vehicle_id (Primary Key), make, model, year, odometer_reading, last_maintenance_date
print('Generating Vehicles...')
vehicles_data = []
makes_models = {'Ford': ['Transit', 'F-250'], 'Mercedes': ['Sprinter'], 'Ram': ['ProMaster', '2500']}

for vehicle_id in range(1, 51):
    make = random.choice(list(makes_models.keys()))
    model = random.choice(makes_models[make])

    # Inject "Dirt": Leave ~10% of last_maintenance_date fields blank (Null)
    if random.random() < 0.010:
        maintenance_date = np.nan
    else:
        maintenance_date = fake.date_between(start_date='-1y', end_date='today')

    vehicles_data.append({
        'vehicle_id': vehicle_id,
        'make': make,
        'model': model,
        'year': random.randint(2015, 2023),
        'odometer_reading': random.randint(10000, 150000),
        'last_maintenance_date': maintenance_date
    })
df_vehicles = pd.DataFrame(vehicles_data)

Generating Vehicles...


In [12]:
# --- 3. Generate delivery_routes ---
# Schema: route_id (Primary Key), driver_id (Foriegn Key), vehicle_id (Foreign Key), dispatch_timestamp,
# delivery_timestamp, distance_miles, fuel_used_gallons.

# --- 3. GENERATE DELIVERY ROUTES (10,000 records) ---
print("Generating Delivery Routes...")
routes_data = []

for route_id in range(1, 10001):
    # Base route info
    driver = random.randint(1, 100)
    vehicle = random.randint(1, 50)
    distance = round(random.uniform(5.0, 300.0), 2)
    
    # Calculate realistic fuel used (assuming avg 10-15 mpg)
    realistic_mpg = random.uniform(10.0, 15.0)
    fuel_used = round(distance / realistic_mpg, 2)
    
    # Generate Timestamps
    dispatch_time = fake.date_time_between(start_date='-1y', end_date='now')
    # Add between 30 mins and 8 hours for delivery
    duration = timedelta(minutes=random.randint(30, 480)) 
    delivery_time = dispatch_time + duration

    # --- INJECTING THE "DIRT" ---
    
    # Dirt 1: 5% of Fuel_Used_Gallons are negative or massive outliers
    anomaly_roll = random.random()
    if anomaly_roll < 0.025: # 2.5% chance of negative fuel
        fuel_used = -abs(fuel_used)
    elif anomaly_roll < 0.05: # 2.5% chance of massive outlier (e.g., 5000 gallons)
        fuel_used = round(random.uniform(1000, 5000), 2)
        
    # Dirt 2: 3% of Delivery_Timestamps occur BEFORE Dispatch_Timestamp
    if random.random() < 0.03:
        delivery_time = dispatch_time - timedelta(minutes=random.randint(10, 120))

    routes_data.append({
        'route_id': route_id,
        'driver_id': driver,
        'vehicle_id': vehicle,
        'dispatch_timestamp': dispatch_time,
        'delivery_timestamp': delivery_time,
        'distance_miles': distance,
        'fuel_used_gallons': fuel_used
    })

df_routes = pd.DataFrame(routes_data)




Generating Delivery Routes...


In [8]:
# --- 4. EXPORT TO CSV ---
print("Exporting to CSV...")
df_drivers.to_csv('drivers_raw.csv', index=False)
df_vehicles.to_csv('vehicles_raw.csv', index=False)
df_routes.to_csv('delivery_routes_raw.csv', index=False)

print("Data generation complete! You now have three raw CSV files ready for PostgreSQL.")

Exporting to CSV...
Data generation complete! You now have three raw CSV files ready for PostgreSQL.


In [17]:
# Explority Analysis of Drivers table for snapshot of data

print("Structural Dimension (rows, columns")
print("------------------------------------")
print(df_drivers.shape)
print("------------------------------------")
print("Inspection of first 10 rows")
print("------------------------------------")
print(df_drivers.head())
print("------------------------------------")
print("Data types, non-null counts")
print("------------------------------------")
print(df_drivers.info())


Structural Dimension (rows, columns
------------------------------------
(100, 6)
------------------------------------
Inspection of first 10 rows
------------------------------------
   driver_id first_name last_name license_class   hire_date  status
0          1   Danielle   Johnson       Class C  2023-11-15  Active
1          2       John    Taylor       Class A  2022-05-28  Active
2          3      Erica   Mcclain       Class A  2022-04-10  Active
3          4   Brittany   Johnson       Class A  2022-05-01  Active
4          5    Jeffery    Wagner       Class C  2021-10-04  Active
------------------------------------
Data types, non-null counts
------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   driver_id      100 non-null    int64 
 1   first_name     100 non-null    object
 2   last_name      100 non-nul

In [18]:
# Explority Analysis of VEHICLES table for snapshot of data

print("Structural Dimension (rows, columns")
print("------------------------------------")
print(df_vehicles.shape)
print("------------------------------------")
print("Inspection of first 10 rows")
print("------------------------------------")
print(df_vehicles.head())
print("------------------------------------")
print("Data types, non-null counts")
print("------------------------------------")
print(df_vehicles.info())

Structural Dimension (rows, columns
------------------------------------
(50, 6)
------------------------------------
Inspection of first 10 rows
------------------------------------
   vehicle_id      make      model  year  odometer_reading  \
0           1       Ram  ProMaster  2015             96619   
1           2      Ford    Transit  2022             66160   
2           3       Ram  ProMaster  2022             73700   
3           4  Mercedes   Sprinter  2016            122997   
4           5  Mercedes   Sprinter  2015             35799   

  last_maintenance_date  
0            2026-04-18  
1            2025-11-24  
2            2025-10-01  
3            2026-04-06  
4            2026-03-17  
------------------------------------
Data types, non-null counts
------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 -----

In [20]:
# Explority Analysis of DELIVERY ROUTES table for snapshot of data

print("Structural Dimension (rows, columns")
print("------------------------------------")
print(df_routes.shape)
print("------------------------------------")
print("Inspection of first 10 rows")
print("------------------------------------")
print(df_routes.head())
print("------------------------------------")
print("Data types, non-null counts")
print("------------------------------------")
print(df_routes.info())
print("------------------------------------")
print(df_routes.isna().sum())

Structural Dimension (rows, columns
------------------------------------
(10000, 7)
------------------------------------
Inspection of first 10 rows
------------------------------------
   route_id  driver_id  vehicle_id  dispatch_timestamp  delivery_timestamp  \
0         1         52          44 2026-03-17 11:46:16 2026-03-17 12:30:16   
1         2         54          34 2025-07-31 21:05:58 2025-08-01 01:42:58   
2         3         16          22 2026-02-12 22:51:40 2026-02-13 02:34:40   
3         4          7           9 2026-02-10 18:04:56 2026-02-10 20:44:56   
4         5          8          43 2026-07-03 02:12:55 2026-07-03 05:58:55   

   distance_miles  fuel_used_gallons  
0          253.88              18.87  
1          204.82              18.76  
2          235.92              21.27  
3          285.35              23.46  
4          286.88              24.83  
------------------------------------
Data types, non-null counts
------------------------------------
<class 'p